# FinGuard - exploratory analysis of the synthetic dataset

Run `make pipeline-local` (or `python -m simulator && python -m features build`) first. This notebook is for exploration only; everything production-relevant lives in the Python packages.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from common.config import get_settings
s = get_settings()
tx = pd.read_parquet(s.path('synthetic_dir') / 'transactions.parquet')
labels = pd.read_parquet(s.path('synthetic_dir') / 'transaction_labels.parquet')
df = tx.merge(labels, on='transaction_id')
df.shape

## Heavy-tailed amounts per transaction type

In [ ]:
df.groupby('transaction_type')['amount'].describe(percentiles=[.5, .9, .99])

## Fraud scenarios: volume, typical amount, failure rate

In [ ]:
(df[df.is_fraud].groupby('fraud_scenario')
   .agg(n=('amount', 'size'), median_amount=('amount', 'median'),
        failed=('status', lambda s: (s == 'FAILED').mean()))
   .sort_values('n', ascending=False))

## Overlap: legitimate traffic also has night activity and new devices

In [ ]:
hour = (df.event_time.dt.hour + 6) % 24
pd.crosstab(hour < 5, df.is_fraud, normalize='columns')

## Point-in-time features: fraud vs. normal (medians)

In [ ]:
f = pd.read_parquet(s.path('processed_dir') / 'transaction_features.parquet').merge(labels, on='transaction_id')
f.groupby('is_fraud')[['amount_to_avg_ratio', 'txn_count_5m', 'is_new_device', 'is_new_recipient',
                       'receiver_in_degree', 'accounts_per_device', 'amount_to_balance_ratio']].mean().T

## Label delay

In [ ]:
fr = df[df.is_fraud].dropna(subset=['fraud_reported_at'])
((fr.fraud_reported_at - fr.event_time).dt.total_seconds() / 86400).describe()